# Hello World
## Example in Zeppelin

This notebook runs in Apache Zeppelin (e.g. on an HDInsight cluster). It reads HVAC sensor sample data from the cluster storage, explores it with Spark DataFrames, and queries it with SQL.

### Imports

PySpark and Spark SQL modules. The **%livy2.pyspark** interpreter runs this paragraph on the cluster's Spark session.

In [1]:
%livy2.pyspark
from pyspark.sql import *
from pyspark.sql.types import *

**Import** CSV and string I/O utilities (for working with raw text or CSV).

In [2]:
%livy2.pyspark
import csv
import io
from io import StringIO

**Read the HVAC CSV** from the cluster path into a Spark DataFrame. First row as header, schema inferred.

In [3]:
%livy2.pyspark
csvFile = spark.read.csv('/HdiSamples/HdiSamples/SensorSampleData/hvac/HVAC.csv', header=True, inferSchema=True)


**Row count:** how many records in the DataFrame.

In [4]:
%livy2.pyspark
# How many rows in dataframe?
csvFile.count()

### HVAC dataset description

The **HVAC** (Heating, Ventilation, and Air Conditioning) sample data contains sensor readings from buildings: date and time, target and actual temperatures, system identifier, system age, and building ID. Below we explore the schema and a sample of the data before registering a temp view for SQL.

In [ ]:
%livy2.pyspark
# Show the schema: column names and types inferred by Spark
csvFile.printSchema()

In [ ]:
%livy2.pyspark
# Preview the first 10 rows
csvFile.show(10)

In [ ]:
%livy2.pyspark
# Summary statistics for numeric columns
csvFile.describe().show()

### Register the DataFrame as a SQL temp view

We register `csvFile` as a temporary view so we can query it with Spark SQL in the following paragraphs.

In [5]:
%livy2.pyspark
csvFile.createOrReplaceTempView("hvac_ss01shh")

**Note on SQL:** Here we use **spark.sql("...").show()** — the same API as in **Google Colab** (in Colab you use the same calls without the %livy2.pyspark line). In Zeppelin you can also use a **%livy2.sql** (or **%sql**) paragraph and write raw SQL; that is similar to **%%sql** in HDInsight. Both run the same SQL against the same Spark session.

**Verify row count** with SQL (should match the DataFrame count above).

In [6]:
%livy2.pyspark
spark.sql("SELECT count(*) from hvac_ss01shh").show()

**Describe the table** to see column names and types.

In [7]:
%livy2.pyspark
spark.sql("DESCRIBE TABLE hvac_ss01shh").show()

**Example query:** temperature difference (target − actual) for a given date. Positive = heating; negative = cooling.

In [8]:
%livy2.pyspark
spark.sql("SELECT `buildingID`, (targettemp - actualtemp) AS `temp_diff`, date FROM hvac_ss01shh WHERE date = '6/1/13'").show()